# Fine-tune on Google Colab

One-click training: **Runtime → Change runtime type → A100**, then **Runtime → Run all**.

| GPU | VRAM | Max Model | Batch | Seq |
|-----|------|-----------|-------|-----|
| T4 | 16 GB | 7B-4bit | 1 | 2048 |
| L4 | 24 GB | 14B-4bit | 1 | 2048 |
| A100 40GB | 40 GB | 32B-4bit | 1 | 4096 |
| A100 80GB | 80 GB | 32B-4bit | 4 | 4096 |

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Clone repo, install CUDA dependencies, verify GPU.

# — Clone —
import subprocess, os
REPO = "https://github.com/piotrjanik/ai-tuner.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
if not os.path.exists("ai-tuner"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd ai-tuner

# — Install —
!pip install -q -r src/requirements-cuda.txt --no-build-isolation

# — GPU check —
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → GPU"
props = torch.cuda.get_device_properties(0)
print(f"\n✓ GPU: {props.name} ({props.total_mem / 1e9:.0f} GB)")
print(f"✓ Flash Attention 2: {'yes' if props.major >= 8 else 'no (eager fallback)'}")

In [ ]:
# 5. Prepare training data (same as `task data` — runs on CPU)
!python src/data/prepare_data.py --config config.yaml

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Prepares training data from config.yaml sources, then runs QLoRA training.
#@markdown Auto-tune picks batch size, seq length, and grad checkpoint based on your GPU.

!python src/data/prepare_data.py --config config.yaml
!python src/train/train_cuda.py --config config.yaml

In [ ]:
# 8. Push adapters to HuggingFace Hub
# !python src/cli.py push

In [ ]:
# 9. (Optional) Download adapters to use locally with MLX export
# !zip -r /content/adapters.zip output/adapters/
# from google.colab import files
# files.download('/content/adapters.zip')